# Tech Challenge Fase 2
## 05.0 — Data Quality Orquestrador

Centraliza regras, criticidades, tolerâncias, caminhos e metadados de qualidade para Bronze, Silver, Gold e Streaming.

Saídas:

```text
config/quality_metadata
logs/data_quality/bronze
logs/data_quality/silver
logs/data_quality/gold
logs/data_quality/streaming
logs/data_quality/summary
logs/data_quality/dashboard
```

## 1. Contexto

A qualidade é transversal à Arquitetura Medalhão:

- Bronze: integridade técnica;
- Silver: validade, padronização e consistência;
- Gold: confiabilidade analítica;
- Streaming: integridade e validade dos eventos.

## 2. Imports

In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, BooleanType
)

## 3. Configuração oficial do projeto

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

BRONZE_PATH = config["paths"]["bronze_path"]
SILVER_PATH = config["paths"]["silver_path"]
GOLD_PATH = config["paths"]["gold_path"]
STREAMING_PATH = config["paths"]["streaming_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

QUALITY_ROOT_PATH = f"{LOG_PATH}/data_quality"
QUALITY_BRONZE_PATH = f"{QUALITY_ROOT_PATH}/bronze"
QUALITY_SILVER_PATH = f"{QUALITY_ROOT_PATH}/silver"
QUALITY_GOLD_PATH = f"{QUALITY_ROOT_PATH}/gold"
QUALITY_STREAMING_PATH = f"{QUALITY_ROOT_PATH}/streaming"
QUALITY_SUMMARY_PATH = f"{QUALITY_ROOT_PATH}/summary"
QUALITY_DASHBOARD_PATH = f"{QUALITY_ROOT_PATH}/dashboard"

print("QUALITY_ROOT_PATH:", QUALITY_ROOT_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Criação dos diretórios

In [0]:
for path in [
    QUALITY_ROOT_PATH,
    QUALITY_BRONZE_PATH,
    QUALITY_SILVER_PATH,
    QUALITY_GOLD_PATH,
    QUALITY_STREAMING_PATH,
    QUALITY_SUMMARY_PATH,
    QUALITY_DASHBOARD_PATH,
    f"{QUALITY_ROOT_PATH}/rejected",
    f"{QUALITY_ROOT_PATH}/history"
]:
    dbutils.fs.mkdirs(path)

print("Estrutura Data Quality criada/validada.")

## 5. Função para registrar regras

Cada regra possui:

- camada e dataset;
- identificador;
- tipo;
- coluna;
- criticidade;
- tolerância;
- indicação de bloqueio;
- descrição.

In [0]:
quality_rules = []

def add_rule(
    layer,
    dataset,
    rule_id,
    rule_name,
    rule_type,
    column_name="",
    reference_dataset="",
    reference_column="",
    severity="MEDIUM",
    tolerance_percent=0.0,
    blocking=False,
    enabled=True,
    description=""
):
    quality_rules.append({
        "layer": str(layer),
        "dataset": str(dataset),
        "rule_id": str(rule_id),
        "rule_name": str(rule_name),
        "rule_type": str(rule_type),
        "column_name": str(column_name),
        "reference_dataset": str(reference_dataset),
        "reference_column": str(reference_column),
        "severity": str(severity),
        "tolerance_percent": float(tolerance_percent),
        "blocking": bool(blocking),
        "enabled": bool(enabled),
        "description": str(description)
    })

## 6. Regras Bronze

In [0]:
for dataset in ["alunos","estados","municipios","metas_municipios","metas_ufs"]:
    add_rule(
        "bronze", dataset,
        f"BRZ_{dataset.upper()}_001",
        "particao_existente",
        "path_exists",
        severity="CRITICAL",
        blocking=True,
        description="Valida a existência da partição Bronze."
    )

    add_rule(
        "bronze", dataset,
        f"BRZ_{dataset.upper()}_002",
        "dataset_nao_vazio",
        "row_count_min",
        severity="CRITICAL",
        blocking=True,
        description="Valida se a partição possui registros."
    )

    add_rule(
        "bronze", dataset,
        f"BRZ_{dataset.upper()}_003",
        "metadados_tecnicos_presentes",
        "required_columns",
        column_name="_ingestion_timestamp,_source_file,_pipeline_step",
        severity="HIGH",
        description="Valida metadados técnicos da ingestão."
    )

## 7. Regras Silver

In [0]:
add_rule(
    "silver","alunos","SLV_ALUNOS_001",
    "id_aluno_nulo","not_null",
    "ID_ALUNO",severity="CRITICAL",
    blocking=True,
    description="A Silver analítica de alunos deve conter somente registros com ID_ALUNO preenchido; registros inválidos devem ser preservados na área de rejeitados."
)

add_rule(
    "silver","alunos","SLV_ALUNOS_002",
    "codigo_municipio_nulo","not_null",
    "CO_MUNICIPIO",severity="CRITICAL",
    blocking=True,
    description="A Silver analítica de alunos deve conter somente registros com CO_MUNICIPIO preenchido; registros inválidos devem ser preservados na área de rejeitados."
)

add_rule(
    "silver","alunos","SLV_ALUNOS_003",
    "duplicidade_aluno_ano","unique",
    "ANO,ID_ALUNO",severity="HIGH",
    description="Valida unicidade do aluno por ano."
)

add_rule(
    "silver","alunos","SLV_ALUNOS_004",
    "alfabetizado_valido","allowed_values",
    "IN_ALFABETIZADO",severity="HIGH",
    description="Aceita apenas 0 ou 1."
)

for dataset in ["municipios","metas_municipios"]:
    add_rule(
        "silver",dataset,
        f"SLV_{dataset.upper()}_001",
        "codigo_municipio_nulo","not_null",
        "CO_MUNICIPIO",severity="CRITICAL",
        blocking=True,
        description="Código municipal obrigatório."
    )

    add_rule(
        "silver",dataset,
        f"SLV_{dataset.upper()}_002",
        "duplicidade_municipio_ano","unique",
        "ANO,CO_MUNICIPIO",severity="HIGH",
        description="Unicidade por ano e município."
    )

for dataset in ["estados","metas_ufs"]:
    add_rule(
        "silver",dataset,
        f"SLV_{dataset.upper()}_001",
        "codigo_uf_nulo","not_null",
        "CO_UF",severity="CRITICAL",
        blocking=True,
        description="Código da UF obrigatório na base estadual; o agregado Brasil deve ser persistido separadamente e não participar desta validação."
    )

    add_rule(
        "silver",dataset,
        f"SLV_{dataset.upper()}_002",
        "duplicidade_uf_ano","unique",
        "ANO,CO_UF",severity="HIGH",
        description="Unicidade por ano e UF."
    )

for dataset in ["municipios","estados","metas_municipios","metas_ufs"]:
    add_rule(
        "silver",dataset,
        f"SLV_{dataset.upper()}_003",
        "percentual_fora_faixa","between",
        "PC_ALUNO_ALFABETIZADO",
        severity="HIGH",
        description="Percentual deve estar entre 0 e 100."
    )

### Tratamento das regras críticas da camada Silver

As regras críticas permanecem com:

```text
severity = CRITICAL
tolerance_percent = 0
blocking = true
```

A aprovação dessas regras será garantida por meio da separação correta entre:

- registros válidos da Silver analítica;
- registros rejeitados;
- agregados de granularidade distinta, como o registro Brasil.

As regras não serão flexibilizadas para ocultar problemas de qualidade.

## 8. Regras Gold

In [0]:
add_rule(
    "gold","gold_alunos","GLD_ALUNOS_001",
    "unicidade_municipio_ano","unique",
    "ANO,CO_MUNICIPIO",
    severity="CRITICAL",
    blocking=True,
    description="A Gold Alunos deve possuir exatamente uma linha por ANO + CO_MUNICIPIO após agregação por chaves técnicas."
)

add_rule(
    "gold","gold_municipios","GLD_MUNICIPIOS_001",
    "unicidade_municipio_ano","unique",
    "ANO,CO_MUNICIPIO",
    severity="CRITICAL",
    blocking=True,
    description="A Gold Municípios deve possuir exatamente uma linha por ANO + CO_MUNICIPIO; joins muitos-para-muitos não são permitidos."
)

add_rule(
    "gold","gold_municipios","GLD_MUNICIPIOS_002",
    "cobertura_join_metas","coverage",
    "META_FINAL_2030",
    severity="HIGH",
    tolerance_percent=1.0,
    description="Até 1% de municípios sem meta."
)

add_rule(
    "gold","gold_estados","GLD_ESTADOS_001",
    "unicidade_uf_ano","unique",
    "ANO,CO_UF",
    severity="CRITICAL",
    blocking=True,
    description="A Gold Estados deve possuir exatamente uma linha por ANO + CO_UF."
)

add_rule(
    "gold","gold_machine_learning","GLD_ML_001",
    "completude_features","completeness",
    "registro_completo_modelo",
    severity="HIGH",
    tolerance_percent=10.0,
    description="Até 10% de registros incompletos."
)

add_rule(
    "gold","gold_machine_learning","GLD_ML_002",
    "target_valido","allowed_values",
    "risco_nao_atingir_meta",
    severity="CRITICAL",
    blocking=True,
    description="Target deve ser 0, 1 ou nulo."
)

## 9. Regras Streaming

In [0]:
add_rule(
    "streaming","eventos","STR_EVENTOS_001",
    "event_id_nulo","not_null",
    "event_id",severity="CRITICAL",
    blocking=True,
    description="Todo evento deve possuir ID."
)

add_rule(
    "streaming","eventos","STR_EVENTOS_002",
    "event_id_duplicado","unique",
    "event_id",severity="HIGH",
    description="Valida unicidade do evento."
)

add_rule(
    "streaming","eventos","STR_EVENTOS_003",
    "valor_fora_intervalo","between",
    "valor",severity="HIGH",
    description="Valor deve estar entre 0 e 100."
)

add_rule(
    "streaming","eventos","STR_EVENTOS_004",
    "chave_territorial_nula","not_null",
    "co_uf,co_municipio",
    severity="CRITICAL",
    blocking=True,
    description="Eventos devem possuir UF e município."
)

## 10. Criação da quality_metadata

In [0]:
schema_quality = StructType([
    StructField("layer", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("rule_id", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("rule_type", StringType(), False),
    StructField("column_name", StringType(), True),
    StructField("reference_dataset", StringType(), True),
    StructField("reference_column", StringType(), True),
    StructField("severity", StringType(), False),
    StructField("tolerance_percent", DoubleType(), False),
    StructField("blocking", BooleanType(), False),
    StructField("enabled", BooleanType(), False),
    StructField("description", StringType(), True)
])

df_quality_metadata = spark.createDataFrame(
    quality_rules,
    schema=schema_quality
)

display(
    df_quality_metadata
    .orderBy("layer","dataset","rule_id")
)

## 11. Validação dos metadados

In [0]:
valid_severities = ["LOW","MEDIUM","HIGH","CRITICAL"]

metadata_validation = (
    df_quality_metadata
    .withColumn(
        "rule_id_count",
        F.count("*").over(
            Window.partitionBy("rule_id")
        )
    )
    .withColumn(
        "rule_id_unique",
        F.col("rule_id_count") == 1
    )
    .withColumn(
        "severity_valid",
        F.col("severity").isin(valid_severities)
    )
    .withColumn(
        "tolerance_valid",
        F.col("tolerance_percent").between(0.0,100.0)
    )
    .withColumn(
        "mandatory_fields_valid",
        F.col("layer").isNotNull()
        & F.col("dataset").isNotNull()
        & F.col("rule_id").isNotNull()
        & F.col("rule_name").isNotNull()
        & F.col("rule_type").isNotNull()
    )
    .withColumn(
        "metadata_valid",
        F.col("rule_id_unique")
        & F.col("severity_valid")
        & F.col("tolerance_valid")
        & F.col("mandatory_fields_valid")
    )
)

display(
    metadata_validation
    .orderBy("layer","dataset","rule_id")
)

## 12. Persistência e resumo

In [0]:
quality_metadata_path = f"{CONFIG_PATH}/quality_metadata"

(
    df_quality_metadata
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression","snappy")
    .save(quality_metadata_path)
)

df_quality_summary = (
    df_quality_metadata
    .groupBy("layer","severity")
    .agg(
        F.count("*").alias("quantidade_regras"),
        F.sum(
            F.when(F.col("blocking"),1).otherwise(0)
        ).alias("regras_bloqueantes"),
        F.sum(
            F.when(F.col("enabled"),1).otherwise(0)
        ).alias("regras_ativas")
    )
)

summary_path = (
    f"{QUALITY_SUMMARY_PATH}/"
    f"quality_rules_config_execution_date={EXECUTION_DATE}"
)

(
    df_quality_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression","snappy")
    .save(summary_path)
)

display(df_quality_summary.orderBy("layer","severity"))

print("quality_metadata salva em:", quality_metadata_path)
print("Resumo salvo em:", summary_path)

## 13. Checklist final

In [0]:
invalid_count = (
    metadata_validation
    .filter(~F.col("metadata_valid"))
    .count()
)

enabled_count = (
    df_quality_metadata
    .filter(F.col("enabled"))
    .count()
)

if invalid_count > 0:
    display(
        metadata_validation
        .filter(~F.col("metadata_valid"))
    )
    raise Exception(
        f"Foram encontradas {invalid_count} "
        f"regras com metadados inválidos."
    )

if enabled_count == 0:
    raise Exception("Nenhuma regra de qualidade está ativa.")

print("Data Quality Orquestrador concluído com sucesso.")
print("Regras ativas:", enabled_count)

## Resultado esperado

```text
config/quality_metadata
logs/data_quality/summary/quality_rules_config_execution_date=YYYY-MM-DD
```

Próximo notebook:

```text
05_1_quality_bronze
```